In [6]:
import numpy as np
import matplotlib.pyplot as plt
import requests
import pandas as pd

In [7]:

import json
import pandas as pd

url = "https://SDMDataAccess.sc.egov.usda.gov/Tabular/post.rest"

def query_soil(sql, label=""):
    response = requests.post(
        url,
        data={"query": sql, "format": "JSON+COLUMNNAME"},
        timeout=30
    )
    if response.status_code == 200:
        data = response.json()
        # Convert to DataFrame
        cols = data["Table"][0]
        rows = data["Table"][1:]
        df = pd.DataFrame(rows, columns=cols)
        print(f"✅ {label} — {len(df)} rows returned")
        return df
    else:
        print(f"❌ Error {response.status_code}:", response.text)
        return None


# ─────────────────────────────────────────
# 1. Survey area list (sacatalog is the correct table!)
# ─────────────────────────────────────────
indiana_areas = query_soil("""
    SELECT areasymbol, areaname, saverest
    FROM sacatalog
    WHERE areasymbol LIKE 'IN%'
""", "Indiana Survey Areas")

iowa_areas = query_soil("""
    SELECT areasymbol, areaname, saverest
    FROM sacatalog
    WHERE areasymbol LIKE 'IA%'
""", "Iowa Survey Areas")


# ─────────────────────────────────────────
# 2. Soil components — taxonomy & drainage
# ─────────────────────────────────────────
indiana_components = query_soil("""
    SELECT l.areasymbol, m.musym, m.muname,
           c.compname, c.comppct_r, c.drainagecl,
           c.taxorder, c.taxsuborder
    FROM legend l
    INNER JOIN mapunit m ON m.lkey = l.lkey
    INNER JOIN component c ON c.mukey = m.mukey
    WHERE l.areasymbol LIKE 'IN%'
    AND c.majcompflag = 'Yes'
""", "Indiana Components")

iowa_components = query_soil("""
    SELECT l.areasymbol, m.musym, m.muname,
           c.compname, c.comppct_r, c.drainagecl,
           c.taxorder, c.taxsuborder
    FROM legend l
    INNER JOIN mapunit m ON m.lkey = l.lkey
    INNER JOIN component c ON c.mukey = m.mukey
    WHERE l.areasymbol LIKE 'IA%'
    AND c.majcompflag = 'Yes'
""", "Iowa Components")


# ─────────────────────────────────────────
# 3. Soil horizon properties (pH, OM, texture)
# ─────────────────────────────────────────
indiana_horizons = query_soil("""
    SELECT l.areasymbol, c.compname,
           ch.hzdept_r, ch.hzdepb_r,
           ch.sandtotal_r, ch.silttotal_r, ch.claytotal_r,
           ch.om_r, ch.ph1to1h2o_r, ch.awc_r
    FROM legend l
    INNER JOIN mapunit m ON m.lkey = l.lkey
    INNER JOIN component c ON c.mukey = m.mukey
    INNER JOIN chorizon ch ON ch.cokey = c.cokey
    WHERE l.areasymbol LIKE 'IN%'
    AND c.majcompflag = 'Yes'
""", "Indiana Horizons")

iowa_horizons = query_soil("""
    SELECT l.areasymbol, c.compname,
           ch.hzdept_r, ch.hzdepb_r,
           ch.sandtotal_r, ch.silttotal_r, ch.claytotal_r,
           ch.om_r, ch.ph1to1h2o_r, ch.awc_r
    FROM legend l
    INNER JOIN mapunit m ON m.lkey = l.lkey
    INNER JOIN component c ON c.mukey = m.mukey
    INNER JOIN chorizon ch ON ch.cokey = c.cokey
    WHERE l.areasymbol LIKE 'IA%'
    AND c.majcompflag = 'Yes'
""", "Iowa Horizons")


# ─────────────────────────────────────────
# 4. Save to CSV
# ─────────────────────────────────────────
for df, name in [
    (indiana_areas,      "indiana_areas"),
    (iowa_areas,         "iowa_areas"),
    (indiana_components, "indiana_components"),
    (iowa_components,    "iowa_components"),
    (indiana_horizons,   "indiana_horizons"),
    (iowa_horizons,      "iowa_horizons"),
]:
    if df is not None:
        df.to_csv(f"{name}.csv", index=False)
        print(f"💾 Saved {name}.csv")

✅ Indiana Survey Areas — 92 rows returned
✅ Iowa Survey Areas — 99 rows returned
✅ Indiana Components — 8838 rows returned
✅ Iowa Components — 12624 rows returned
✅ Indiana Horizons — 31395 rows returned
✅ Iowa Horizons — 52478 rows returned
💾 Saved indiana_areas.csv
💾 Saved iowa_areas.csv
💾 Saved indiana_components.csv
💾 Saved iowa_components.csv
💾 Saved indiana_horizons.csv
💾 Saved iowa_horizons.csv


In [10]:
# Run this cell FIRST before anything else
import requests
import time
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (mean_squared_error, r2_score,
                             accuracy_score, classification_report,
                             confusion_matrix)
from sklearn.impute import SimpleImputer
warnings.filterwarnings("ignore")

print("✅ All imports successful")

✅ All imports successful


In [11]:
NOAA_TOKEN = "vbaZNBvveZyVQEaXZeeOnTXFcpKtiNZw"
YEARS      = list(range(2010, 2024))
STATES     = {"Indiana": "FIPS:18", "Iowa": "FIPS:19"}
 
plt.rcParams.update({
    "figure.facecolor": "#F7F9FC", "axes.facecolor": "#F7F9FC",
    "axes.spines.top": False, "axes.spines.right": False,
})
C = {"indiana":"#2E86AB","iowa":"#A23B72","corn":"#F18F01",
     "teal":"#44BBA4","red":"#E94F37","purple":"#7F77DD"}
 
print("=" * 65)
print("  APPROACH 2: SOIL + WEATHER + YIELD PIPELINE")
print("=" * 65)

  APPROACH 2: SOIL + WEATHER + YIELD PIPELINE


In [ ]:
NOAA_BASE   = "https://www.ncdc.noaa.gov/cdo-web/api/v2/data"
NOAA_HEADERS = {"token": NOAA_TOKEN}
 
# Growing season = April–September (months 4–9)
GROW_START = "-04-01"
GROW_END   = "-09-30"
 
def noaa_fetch(dataset, datatype, location_id, year, limit=1000):
    """Fetch a single NOAA data query with retry logic."""
    params = {
        "datasetid":  dataset,
        "datatypeid": datatype,
        "locationid": location_id,
        "startdate":  f"{year}{GROW_START}",
        "enddate":    f"{year}{GROW_END}",
        "units":      "standard",
        "limit":      limit,
    }
    for attempt in range(3):
        try:
            r = requests.get(NOAA_BASE, headers=NOAA_HEADERS, params=params, timeout=30)
            if r.status_code == 200:
                results = r.json().get("results", [])
                return results
            elif r.status_code == 429:
                print(f"    Rate limited — waiting 5s...")
                time.sleep(5)
            else:
                break
        except Exception as e:
            time.sleep(2)
    return []
 
def fetch_state_weather(state_name, fips_id):
    """Fetch annual growing-season weather for a state, all years."""
    rows = []
    print(f"  Fetching {state_name}...")
    for year in YEARS:
        # Precipitation — total growing season
        precip_raw = noaa_fetch("GHCND", "PRCP", fips_id, year)
        precip_mm  = (sum(r["value"] for r in precip_raw) / 10.0
                      if precip_raw else np.nan)
 
        # Tmax — average summer max temp
        tmax_raw  = noaa_fetch("GHCND", "TMAX", fips_id, year)
        tmax_c    = (np.mean([r["value"] for r in tmax_raw]) / 10.0
                     if tmax_raw else np.nan)
        tmax_f    = (tmax_c * 9/5 + 32) if not np.isnan(tmax_c) else np.nan
 
        # Tmin — average spring min temp
        tmin_raw  = noaa_fetch("GHCND", "TMIN", fips_id, year)
        tmin_c    = (np.mean([r["value"] for r in tmin_raw]) / 10.0
                     if tmin_raw else np.nan)
        tmin_f    = (tmin_c * 9/5 + 32) if not np.isnan(tmin_c) else np.nan
 
        # Growing degree days (base 50°F) — simplified from tmax/tmin
        if not (np.isnan(tmax_f) or np.isnan(tmin_f)):
            daily_tavg = (tmax_f + tmin_f) / 2
            gdd = max(0, daily_tavg - 50) * 183   # ~183 growing season days
        else:
            gdd = np.nan
 
        rows.append({
            "state":         state_name,
            "year":          year,
            "precip_mm":     round(precip_mm, 1) if not np.isnan(precip_mm) else np.nan,
            "tmax_f":        round(tmax_f, 1)    if not np.isnan(tmax_f)    else np.nan,
            "tmin_f":        round(tmin_f, 1)    if not np.isnan(tmin_f)    else np.nan,
            "gdd":           round(gdd, 0)        if not np.isnan(gdd)       else np.nan,
        })
        time.sleep(0.2)   # be polite to the API
        print(f"    {year}: precip={precip_mm:.0f}mm  tmax={tmax_f:.1f}°F  gdd={gdd:.0f}", flush=True)
 
    return pd.DataFrame(rows)
 
# Check if token is set
if NOAA_TOKEN == "vbaZNBvveZyVQEaXZeeOnTXFcpKtiNZw":
    print("\n  ⚠️  NOAA token not set — using realistic historical weather fallback.")
    print("  📌 Replace NOAA_TOKEN at the top of this script with your token.")
    print("     Get it free at: https://www.ncdc.noaa.gov/cdo-web/token\n")
 
    # Realistic Indiana & Iowa growing season weather 2010–2023
    # Sources: NOAA Climate Normals + known drought years (2012, 2019)
    weather_data = {
        "Indiana": [
            (2010,760,84,52,2200),(2011,620,86,54,2050),(2012,380,92,58,2650),
            (2013,700,82,50,1950),(2014,680,80,48,1850),(2015,820,81,51,2000),
            (2016,740,83,52,2100),(2017,690,84,53,2150),(2018,750,85,54,2200),
            (2019,480,83,51,2050),(2020,710,84,52,2100),(2021,650,85,53,2180),
            (2022,700,86,54,2250),(2023,730,84,52,2120),
        ],
        "Iowa": [
            (2010,720,83,50,2150),(2011,600,85,52,2000),(2012,350,91,57,2600),
            (2013,680,81,49,1900),(2014,660,79,47,1800),(2015,800,80,50,1950),
            (2016,720,82,51,2050),(2017,670,83,52,2100),(2018,730,84,53,2150),
            (2019,460,82,50,2000),(2020,690,83,51,2050),(2021,630,84,52,2130),
            (2022,680,85,53,2200),(2023,710,83,51,2080),
        ]
    }
    rows = []
    for state, vals in weather_data.items():
        for year, precip, tmax, tmin, gdd in vals:
            rows.append({"state": state, "year": year, "precip_mm": precip,
                         "tmax_f": tmax, "tmin_f": tmin, "gdd": gdd})
    weather_df = pd.DataFrame(rows)
    print(f"  ✅ Fallback weather data loaded — {len(weather_df)} rows")
else:
    frames = []
    for state_name, fips in STATES.items():
        frames.append(fetch_state_weather(state_name, fips))
    weather_df = pd.concat(frames, ignore_index=True)
    weather_df.to_csv("noaa_weather_fetched.csv", index=False)
    print(f"\n  ✅ NOAA data fetched — {len(weather_df)} rows saved to noaa_weather_fetched.csv")
 
print(f"\n  Weather summary:")
print(weather_df.groupby("state")[["precip_mm","tmax_f","tmin_f","gdd"]].mean().round(1).to_string())
 

  Fetching Indiana...
    2010: precip=10mm  tmax=44.5°F  gdd=0
    2011: precip=32mm  tmax=43.4°F  gdd=0
    2012: precip=7mm  tmax=43.5°F  gdd=0
    2013: precip=23mm  tmax=43.1°F  gdd=0
    2014: precip=17mm  tmax=43.4°F  gdd=0
    2015: precip=14mm  tmax=43.7°F  gdd=0
    2016: precip=15mm  tmax=43.3°F  gdd=0
    2017: precip=18mm  tmax=44.2°F  gdd=0
    2018: precip=13mm  tmax=42.2°F  gdd=0
    2019: precip=19mm  tmax=43.4°F  gdd=0
    2020: precip=9mm  tmax=42.9°F  gdd=0
    2021: precip=10mm  tmax=43.6°F  gdd=0
    2022: precip=12mm  tmax=42.8°F  gdd=0
    2023: precip=8mm  tmax=43.5°F  gdd=0
  Fetching Iowa...
    2010: precip=11mm  tmax=44.3°F  gdd=0
    2011: precip=10mm  tmax=42.5°F  gdd=0
    2012: precip=14mm  tmax=43.7°F  gdd=0
    2013: precip=22mm  tmax=41.1°F  gdd=0
